# 07 — Online RL Fine-Tuning of the SFT'd SLM with GRPO · Google Colab

**Phase 3 (RL stage)** · MSc Thesis — ECLIPSE project

---

## How to read this notebook

This notebook is the **online RL stage** that follows the supervised
fine-tuning of notebook 05. It takes the LoRA-SFT'd Gemma-4 E4B and
improves it further by sampling rollouts from the current policy,
scoring them with the real `MERLINReward` from CityLearn, and updating
the LoRA weights using a **GRPO**-style (Group Relative Policy
Optimization) gradient.

**The policy produced here is a Chain-of-Thought (CoT) policy.** At every
step the SLM emits a short `<thought>...</thought>` rationale followed by
the `<action>` tags — the format of `src.agent.make_minimal_prompt`. RL
trains on *all* generated tokens, so it refines both the reasoning and
the actions. This serves thesis RQ3 (interpretable natural-language
rationales).

The notebook is designed to be *read top-to-bottom*. The cells are
structured in three layers:

1. **Conceptual cells (§§ 0.5, 0.6)** — explain GRPO from first principles
   with a worked numerical example *before* you see any heavy code. If
   you have never trained a model with RL, read these very carefully.
2. **Setup cells (§§ 1–6)** — install, mount Drive, load the model.
   Standard Colab boilerplate, kept terse.
3. **Core training cells (§§ 7–11)** — the rollout primitive, the GRPO
   loss, the training loop, evaluation, plots. Each of these is preceded
   by a markdown cell that explains both *what* the code does and *why*
   it does it that way.

If anything in the math feels under-explained, the **companion document
[`docs/GRPO_EXPLAINED.md`](../docs/GRPO_EXPLAINED.md)** is the formal
thesis-style reference. It contains the full derivation, the algorithm
comparison table, the hyperparameter rationale, and an annotated reading
list of the source papers.

---

## ⚠ Precondition — this notebook expects a *CoT-capable* SFT adapter

RL refines a prior; it cannot conjure a behaviour the prior never had.
This notebook feeds the model the **CoT prompt** (`make_minimal_prompt`,
which asks for a `<thought>` block). For that to work, the SFT adapter
from notebook 05 must itself have been trained to **produce** `<thought>`
blocks — i.e.:

- the distillation dataset (nb 04) was **re-cured to include `<thought>`
  rationales** in each row's `response` (however you synthesise them —
  LLM-generated or programmatic), and
- notebook 05 was re-run with `SYSTEM_PROMPT = make_minimal_prompt(...)`
  (not the old no-CoT `make_sft_prompt`).

If the SFT adapter has **zero `<thought>` mass** (i.e. it was trained on
the old no-CoT dataset), the prompt here asks for reasoning the model was
never taught — that is the SFT↔eval prompt drift documented in nb 05
§ 19. The training will still run, but the early `<thought>` blocks will
be poor and the KPI may degrade before it recovers. Re-cure the dataset
first.

---

## Glossary — RL terms in CityLearn vocabulary

| RL term | What it means here, in plain words |
|---|---|
| **Policy** $\pi_\theta$ | The SLM (Gemma-4 + LoRA). Reads the rendered state, outputs an action block. "Parameters $\theta$" = the LoRA weights we update. |
| **State** $s_t$ | The text block that summarises the district at hour $t$: time, price, carbon, plus per-building SoC, load, solar, last net consumption. |
| **Action** $a_t$ | The 3-line action block the SLM emits — one of {CHARGE_100…IDLE…DISCHARGE_100} per building. |
| **Reward** $r_t$ | The negative cost from `MERLINReward` at step $t$. Always ≤ 0; less negative = better. |
| **Return** $R$ | The sum of rewards over a window of $W$ steps. Less negative = better strategy. |
| **Rollout** | One sample trajectory: starting state, action, next state, action, … for $W$ steps. |
| **Trajectory** | Same thing as a rollout in this notebook. |
| **Group** | The set of $G$ rollouts we sample *from the same starting state* in one optimizer update. The group lets us compute a baseline. |
| **Baseline** $b$ | A reference reward level we subtract from each return to reduce variance. In GRPO the baseline is just the mean of the group's returns. |
| **Advantage** $A_g$ | How much *better* rollout $g$ did than the group mean, standardised by the group std. Positive → make this rollout more likely. Negative → less likely. |
| **Log-probability** $\log\pi(c\|s)$ | The model's own estimate of how likely it is to emit the completion $c$ given prompt $s$. Updating the policy = nudging these log-probs up or down. |
| **Reference policy** $\pi_\mathrm{ref}$ | A *frozen* copy of the policy we started from. Used in the KL term to stop the policy from drifting too far. |
| **KL divergence** | A measure of how different the trained policy is from the reference. Small KL = still close to the prior; large KL = wandered far. |
| **$\beta$ (KL coefficient)** | How strongly we penalise drift from $\pi_\mathrm{ref}$. Larger $\beta$ → policy moves less per update. |
| **Fallback / parse failure** | The model emitted something we couldn't parse as an action. We default to IDLE and apply a small penalty. |
| **CoT** | Chain-of-Thought — the `<thought>...</thought>` block the SLM produces before the action tags. The reasoning trace this notebook trains and refines. |

Keep this table open while you read the rest of the notebook. Every time
you see one of these symbols in a math line, mentally translate it into
the right-hand column.

## § 0 — Runtime & Config

> **Edit this cell only.** Nothing else needs changing for a smoke run.

The defaults are **toy-scale**: small group size $G$, short window $W$,
few updates. The whole loop runs in under an hour on a free T4 GPU so you
can verify the plumbing end-to-end before paying for a longer run. Flip
`FULL_SCALE = True` (or override individual constants below) when you
graduate to an A100 for the thesis-scale run.

**Hyperparameter groups, in plain words:**

- **Model / LoRA** — which base model we load and how much trainable
  capacity we attach. Must match nb 05 so the SFT'd adapter is
  shape-compatible.
- **GRPO** — the RL algorithm's knobs: how many rollouts per update, how
  long each rollout is, how many updates total, and how strongly the
  policy is pulled back toward the SFT reference.
- **CoT shaping** — `FALLBACK_PENALTY` discourages malformed output;
  `COT_BONUS` is an *optional* nudge that rewards a well-formed
  `<thought>` block. It is **off by default** — the primary mechanism
  keeping CoT alive is the KL anchor (§ 5), not a hand-tuned bonus.
- **Episode geometry** — when in the simulated year we sample rollouts
  from. The seasonal pool prevents the policy from overfitting to summer.
- **Evaluation slice** — a deterministic, cheap KPI probe we run between
  updates to track progress. Not the final evaluation.

For a deeper explanation of each individual hyperparameter, see § 7
"Hyperparameter rationale" in [`docs/GRPO_EXPLAINED.md`](../docs/GRPO_EXPLAINED.md).

In [ ]:
# CRITICAL: import unsloth before transformers / trl / peft so its kernel
# patches actually attach. Same rule as in nb 05.
try:
    import unsloth  # noqa: F401
except ImportError:
    pass  # § 1 install will follow on a fresh kernel

import os, sys, subprocess, time, warnings, json, random, math, re
from pathlib import Path
import numpy as np

# ── Repo ──────────────────────────────────────────────────────────────────
GITHUB_REPO = "https://github.com/antonisbast/eclipse-thesis.git"
REPO_DIR    = "/content/eclipse-thesis"

# ── SFT'd LoRA adapter (produced by nb 05, MUST be CoT-trained — see Precondition) ─
# Glob is searched after Drive mount in § 3. Newest match wins.
SFT_ADAPTER_GLOB = "/content/drive/MyDrive/eclipse-thesis/sft_adaptersV3/**/lora_adapter"
ALLOW_RL_FROM_BASE = True   # if no SFT adapter found, RL from the bare base Gemma

# ── Model ─────────────────────────────────────────────────────────────────
MODEL_ID:       str  = "unsloth/gemma-4-E4B-it"
LOAD_IN_4BIT:   bool = True
MAX_SEQ_LEN:    int  = 1024
MAX_NEW_TOKENS: int  = 120   # <thought> ~80 + action block ~30 + slack

# ── LoRA — MUST match the SFT'd adapter rank/alpha so it loads cleanly ───
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05   # small dropout helps RL exploration vs the 0.0 used at SFT

# ── GRPO hyperparameters ─────────────────────────────────────────────────
FULL_SCALE = False     # flip True on A100 for thesis-scale run

if FULL_SCALE:
    GROUP_SIZE   = 4    # G — rollouts per update sharing the same t0
    WINDOW_STEPS = 48   # W — one diurnal cycle
    UPDATES      = 300  # total optimizer updates
    EVAL_EVERY   = 25
    CHECKPOINT_EVERY = 25
else:
    GROUP_SIZE       = 2
    WINDOW_STEPS     = 12
    UPDATES          = 8
    EVAL_EVERY       = 4
    CHECKPOINT_EVERY = 4

KL_BETA        = 0.04         # KL(policy ‖ frozen ref). Halve if KL > 20 nats; double if reward collapses.
LEARNING_RATE  = 5e-6         # Unsloth Gemma-4 GRPO default. Raise to 1e-5 after 50 stable updates.
GRAD_CLIP      = 1.0          # StarPO-S stabilizer (RAGEN)
ENTROPY_COEF   = 0.0          # off by default; small (1e-3) helps if rollouts collapse to one token

# ── CoT / output shaping ─────────────────────────────────────────────────
FALLBACK_PENALTY = 0.5        # subtracted from per-step reward when parsing fails (treated as IDLE)
COT_BONUS        = 0.0        # OPTIONAL per-step reward bonus when the completion contains a well-formed
                              # <thought> block. DEFAULT 0.0 = OFF. RL judges CoT only indirectly, via
                              # the KL anchor to a CoT-capable reference (§ 5). Turn this on (small,
                              # e.g. 0.03-0.08) ONLY if the § 11 CoT-health plot shows reasoning decaying.
COT_MIN_WORDS    = 3          # a <thought> shorter than this counts as 'degenerate', not a real rationale

# Sampling at rollout time. Greedy at eval time.
ROLLOUT_TEMPERATURE = 0.7
ROLLOUT_TOP_P       = 0.9

# ── Episode geometry ─────────────────────────────────────────────────────
# t0 is sampled from a seasonal-stratified pool so the policy doesn't overfit to summer.
T0_POOL_SEASONS = {
    "winter": (0,    2160),   # Jan-Mar
    "spring": (2160, 4320),   # Apr-Jun
    "summer": (4320, 6480),   # Jul-Sep
    "autumn": (6480, 8640),   # Oct-Dec — leave headroom for WINDOW_STEPS
}

# ── Evaluation slice (between-update KPI probe; cheap, deterministic) ────
EVAL_START = 4000               # summer week with batteries primed by then
EVAL_LEN   = 168                # 1 week. Set to 8 760 for the full-year eval.

# ── Building slices (Phase 3 single-agent, group-centralized over 3) ─────
N_BUILDINGS    = 3
TRAIN_SLICE    = [0, 1, 2]      # in-distribution
HELDOUT_SLICE  = [3, 4, 5]      # generalization probe (also Phase 4 agent β)

# ── CityLearn ─────────────────────────────────────────────────────────────
CITYLEARN_VERSION = "2.6.0b2"

# ── Output ────────────────────────────────────────────────────────────────
RUN_NAME    = time.strftime("grpo_%Y%m%d_%H%M%S")
DRIVE_ROOT  = "/content/drive/MyDrive/eclipse-thesis"
OUT_DIR     = f"{DRIVE_ROOT}/rl_runs/{RUN_NAME}"

SEED = 42
HF_TOKEN = ""

try:
    import torch
    if torch.cuda.is_available():
        _gpu  = torch.cuda.get_device_name(0)
        _vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✓ GPU: {_gpu}  ({_vram:.1f} GB VRAM)")
    else:
        print("⚠  No GPU — Runtime → Change runtime type → GPU")
except ImportError:
    print("torch not yet installed — will be in § 1")

print(f"Scale: {'FULL' if FULL_SCALE else 'TOY'}  |  G={GROUP_SIZE}  W={WINDOW_STEPS}  updates={UPDATES}")
print(f"CoT shaping: FALLBACK_PENALTY={FALLBACK_PENALTY}  COT_BONUS={COT_BONUS} ({'ON' if COT_BONUS else 'off'})")

## § 0.5 — GRPO in Plain Words

> *Read this section once, slowly. Everything below it is implementation
> detail for what's described here.*

### The 90-second story

The SLM trained in nb 05 by behaviour cloning is a *student* of the SAC
teacher. Wherever the teacher was suboptimal, the student is at best
equally suboptimal. We have no way for SFT to *exceed* the teacher.

RL changes the game. Instead of asking "did you copy the teacher
correctly?", we ask **"did you make CityLearn happy?"** — measured by the
real MERLIN reward summed over a window of hours. The policy is then
nudged so that the actions it took during *good* hours become more likely,
and the actions it took during *bad* hours become less likely.

The hard part is: what does "good" even mean when every reward is
negative? Should an action that earned −3.4 be encouraged or discouraged?

GRPO's answer: **compare it to other actions the policy could have taken
from the same starting state.** If you sample 4 rollouts from the same
starting hour and three earn ~−5 while one earns −3.4, then the −3.4 one
was *good* in this context. That's what we encourage. If three earned
~−2 and one earned −3.4, then −3.4 was bad.

GRPO formalises "good in context" as the **group-normalised advantage**:

$$
A_g \;=\; \frac{R_g - \bar{R}}{\sigma_R + \epsilon}
$$

$\bar R$ is the average return of the group; $\sigma_R$ is the standard
deviation. So $A_g$ is just a *z-score* of rollout $g$'s return within the
group. Positive = better than the group mean. Negative = worse.

### The data flow, as an ASCII diagram

```
                           ┌──────── one update ────────┐
                           │                            │
  sample t0 (random hour) ─┤  build G fresh envs all     │
                           │  starting at t0             │
                           │                            │
                           │  for each of G envs:        │
                           │     for W steps:            │
                           │        render state         │
                           │        SLM → thought+action │
                           │        env.step(action)     │
                           │        store r_t, log_pi    │
                           │                            │
                           │  ─────────────────────────  │
                           │  Returns:   R_1, …, R_G     │
                           │  Mean / std → advantages    │
                           │  A_g = (R_g − R̄) / σ_R       │
                           │  ─────────────────────────  │
                           │  Loss = −E[A_g · log π(c|s)]│
                           │        + β · KL(π ‖ π_ref)  │
                           │  loss.backward()            │
                           │  optimizer.step()           │
                           │                            │
                           └────────────────────────────┘
```

Note that `c` (the completion the log-prob is taken over) is the **whole**
`<thought>…</thought><action>…</action>` block — reasoning tokens
included. RL shapes the rationale and the action together.

### Why two terms in the loss?

1. **Policy-gradient term** $-A_g \log \pi$: this is the "do more of the
   good" / "do less of the bad" engine. By itself it would let the policy
   drift arbitrarily far from where it started — and a far-drifted policy
   tends to produce gibberish.
2. **KL term** $\beta \cdot \mathrm{KL}(\pi \Vert \pi_\mathrm{ref})$: a
   leash. Measures how much the policy has diverged from a frozen
   reference and adds that to the loss. Small $\beta$ = long leash; large
   $\beta$ = short leash.

Without the KL leash, RL on an LLM almost always destroys the model's
language and reasoning abilities within a few dozen updates. The leash is
what makes RL on top of SFT viable — and, in our CoT setup, it is also
the main force that keeps the `<thought>` block from collapsing into
degenerate filler (the reference still produces real rationales, so the
KL term pulls the policy back whenever its thoughts decay).

### Multi-step credit assignment — the windowed adaptation

Classical GRPO (as in DeepSeekMath) operates on *one-shot* problems:
prompt → one completion → one scalar reward. CityLearn is **multi-step**:
every action changes the SoC, the next state depends on the action, the
reward comes in 8,760 hourly pieces per year.

Naïvely treating each hour as a separate "prompt" loses the structure
that battery actions have hours-long consequences. Naïvely treating the
whole year as one trajectory is computationally impossible — one update
would take a full day of inference.

The compromise — borrowed from RAGEN/StarPO 2025 — is to use a **window**
of W steps (W=48 is one diurnal cycle) as the unit of trajectory. The
window-summed return is what gets standardised across the group. Every
action token *within* the window receives the same group advantage. This
is coarse credit assignment, but it is the documented robust default.

## § 0.6 — Numerical Walkthrough (toy demo, runs in ms)

Before we touch CityLearn or the SLM, let's run the GRPO math on hand-
crafted fake numbers. This cell only uses NumPy — no model, no env —
and prints every intermediate quantity so you can see exactly what the
algorithm computes.

**Setup:** $G=2$ rollouts, $W=3$ steps. Per-step rewards and per-token
log-probabilities are fabricated. Rollout 1 is the *relatively better*
one (less negative return). The cell:

1. Computes returns $R_g = \sum_t r_{g,t}$.
2. Computes group mean $\bar R$ and std $\sigma_R$.
3. Computes advantages $A_g = (R_g - \bar R) / (\sigma_R + \epsilon)$.
4. Computes the per-rollout policy-gradient contribution $-A_g \sum_t \log\pi(c_t|s_t)$.
5. Computes the per-rollout KL contribution from the difference between
   policy and reference log-probs.
6. Adds them up to get the final loss.

Then it shows two **failure modes** for intuition:

- *Echo Trap*: all rollouts have the same return → $\sigma_R = 0$ →
  advantages are undefined and the policy-gradient term vanishes.
- *Zero KL*: policy = reference → KL term is 0 → policy can drift freely.

Run this cell first. The numbers it prints are the same numbers that
will appear inside `grpo_loss(...)` in § 8, just on real data.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# GRPO numerical walkthrough — pure NumPy, no model needed.
# Goal: see every intermediate quantity that GRPO computes.
# ─────────────────────────────────────────────────────────────────────────
import numpy as np

EPS  = 1e-6
BETA = 0.04           # same KL coefficient we'll use in real training

# ── Fabricated rollouts ───────────────────────────────────────────────────
# 2 rollouts, 3 steps each. MERLIN rewards are always ≤ 0 (negative cost).
# Rollout 1 sums to -3.40 (the BETTER of the two — less negative).
# Rollout 2 sums to -4.50.
per_step_rewards = np.array([
    [-1.20, -0.80, -1.40],   # rollout 1
    [-1.10, -2.40, -1.00],   # rollout 2
])  # shape (G, W) = (2, 3)

# For each (rollout, step) suppose the SLM emitted ~20 tokens.
# Pretend we already summed log π(c_t | s_t) over those tokens.
# These are NEGATIVE numbers (log-probs of any token sequence sum to negatives).
logp_policy = np.array([
    [-30.0, -28.0, -32.0],   # rollout 1, more confident → higher log-prob (less negative)
    [-35.0, -45.0, -33.0],   # rollout 2
])

# The frozen reference (the SFT'd model) gives slightly different log-probs.
logp_ref = np.array([
    [-32.0, -30.0, -33.0],
    [-34.0, -40.0, -34.0],
])

# Number of tokens generated at each step — needed to convert per-completion
# log-prob sums into per-token averages for the KL.
n_tokens = np.array([
    [25, 22, 28],
    [27, 30, 25],
])

# ── Step 1: returns ───────────────────────────────────────────────────────
returns = per_step_rewards.sum(axis=1)
print("STEP 1 — per-rollout returns")
for g, R in enumerate(returns):
    print(f"   R_{g+1} = sum(rewards) = {R:+.3f}")

# ── Step 2: group statistics ──────────────────────────────────────────────
R_mean = returns.mean()
R_std  = returns.std()
print(f"\nSTEP 2 — group statistics")
print(f"   mean R̄ = {R_mean:+.3f}")
print(f"   std  σ_R = {R_std:.3f}")

# ── Step 3: advantages ────────────────────────────────────────────────────
advantages = (returns - R_mean) / (R_std + EPS)
print(f"\nSTEP 3 — group-normalised advantages")
for g, A in enumerate(advantages):
    sign = "BETTER than group" if A > 0 else "WORSE  than group"
    print(f"   A_{g+1} = (R_{g+1} − R̄) / σ_R = {A:+.3f}   ({sign})")

# ── Step 4: policy-gradient term per (rollout, step) ──────────────────────
# pg_{g,t} = − A_g · log π(c_{g,t} | s_{g,t})
# Minimising this term increases log π for rollouts with A_g > 0 (good ones)
# and decreases log π for A_g < 0 (bad ones).
pg_term = -advantages[:, None] * logp_policy   # broadcast A across W steps
print(f"\nSTEP 4 — policy-gradient terms (− A_g · log π_θ)")
for g in range(2):
    print(f"   rollout {g+1}: {pg_term[g]}")
print(f"   mean over all (g,t):  pg = {pg_term.mean():+.4f}")

# ── Step 5: KL term per (rollout, step), per token ────────────────────────
# Approx KL ≈ (log π_θ − log π_ref).mean()  averaged over completion tokens.
kl_per_step = (logp_policy - logp_ref) / n_tokens   # nats per token
print(f"\nSTEP 5 — KL contributions per step (nats / completion token)")
for g in range(2):
    print(f"   rollout {g+1}: {np.round(kl_per_step[g], 4)}")
print(f"   mean over all (g,t):  KL = {kl_per_step.mean():+.4f}")

# ── Step 6: total loss ────────────────────────────────────────────────────
loss = pg_term.mean() + BETA * kl_per_step.mean()
print(f"\nSTEP 6 — total loss")
print(f"   loss = mean(pg)  +  β · mean(KL)")
print(f"        = {pg_term.mean():+.4f}  +  {BETA} × {kl_per_step.mean():+.4f}")
print(f"        = {loss:+.4f}")
print("   → call loss.backward() and step the optimizer (in real training)")

# ─────────────────────────────────────────────────────────────────────────
# Pathological case 1 — Echo Trap: both rollouts identical → σ_R = 0
# ─────────────────────────────────────────────────────────────────────────
print("\n" + "─"*60)
print("PATHOLOGY — Echo Trap: all rollouts have the same return")
echo_returns    = np.array([-4.0, -4.0])
echo_advantages = (echo_returns - echo_returns.mean()) / (echo_returns.std() + EPS)
print(f"   returns = {echo_returns}")
print(f"   σ_R = {echo_returns.std():.6f}")
print(f"   advantages = {echo_advantages}   ← essentially zero, policy can't learn")
print("   Diagnosis: policy has collapsed to one trajectory. Raise temperature,")
print("              raise KL β, or shorten the window so states differ more.")

# ─────────────────────────────────────────────────────────────────────────
# Pathological case 2 — Zero KL: policy = reference everywhere
# ─────────────────────────────────────────────────────────────────────────
print("\nPATHOLOGY — Zero KL: policy hasn't moved from reference")
zero_kl = (logp_policy - logp_policy) / n_tokens   # reference == policy
print(f"   KL = {zero_kl.mean():+.4f}")
print("   The β·KL term contributes 0 to the loss. The policy is free to move,")
print("   but the leash is currently slack. Watch the KL plot in § 11 to confirm")
print("   the policy actually drifts after the first few updates.")

print("\n" + "─"*60)
print("You have now seen every intermediate quantity that GRPO computes.")
print("§§ 7–9 will do exactly this on real CityLearn rollouts, with torch tensors")
print("in place of these numpy arrays, but the math is identical.")

## § 1 — Install Dependencies

Same recipe as nb 05 — CityLearn 2.6.0b2 pinned (for `evaluate_v2`), Unsloth + TRL pinned to the versions that play with Gemma-4 4-bit + LoRA. We don't strictly need TRL's `GRPOTrainer` here (we write our own loop, see § 8) but we keep TRL installed for the collator utilities and for consistency with the rest of the pipeline.

*Why two install cells and not one?* Pip's resolver gets stuck if you ask it to satisfy CityLearn 2.6.0b2 plus a Unsloth-pinned `transformers` simultaneously. We install CityLearn `--no-deps` first to dodge the conflict, then install Unsloth's stack on top.

In [ ]:
# CityLearn 2.6 (pre-release). --no-deps because it pulls heavy EnergyPlus extras.
!pip install -q numpy gymnasium doe-xstock nrel-pysam
!pip install -q --pre "CityLearn=={CITYLEARN_VERSION}" --no-deps

import citylearn
assert citylearn.__version__.startswith("2.6"), (
    f"Expected CityLearn 2.6.x, got {citylearn.__version__}. "
    f"src.eval depends on evaluate_v2() which only exists in 2.6+."
)
print(f"✓ CityLearn {citylearn.__version__}")

In [ ]:
!pip install -q --upgrade pip
!pip install -q --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps git+https://github.com/unslothai/unsloth-zoo.git
!pip install -q --upgrade transformers peft accelerate bitsandbytes datasets sentencepiece torchao
!pip install -q "trl>=0.15,<0.20"
!pip install -q triton==3.6.0

import trl, transformers, peft
print(f"✓ trl={trl.__version__}  transformers={transformers.__version__}  peft={peft.__version__}")

## § 1.5 — Optional Performance Deps (best-effort)

`hf_transfer` accelerates HF Hub downloads, `xformers` and `flash-attn` provide faster attention kernels. Each install is wrapped in try/except so a failed compile doesn't break the notebook — we just fall back to a slower attention backend.

In [ ]:
import subprocess, sys, importlib

def _try_install(label, pip_args):
    try:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pip_args,
                           capture_output=True, text=True, timeout=600)
        ok = r.returncode == 0
        print(f"{'✓' if ok else '✗'} {label}")
        return ok
    except Exception as e:
        print(f"✗ {label}  ({type(e).__name__}: {e})")
        return False

_try_install("hf_transfer", ["hf_transfer"]); os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
_try_install("xformers",    ["xformers"])
_try_install("flash-attn (best effort)", ["flash-attn", "--no-build-isolation"])

## § 2 — Clone Repo (pulls `src/`)

All shared code (env factory, state renderer, action parser, eval helpers) lives in `src/` in the GitHub repo. Cloning here lets us `from src.env import ...` exactly the way every other notebook does.

In [ ]:
if not os.path.exists(REPO_DIR):
    res = subprocess.run(["git", "clone", GITHUB_REPO, REPO_DIR], capture_output=True, text=True)
    print(res.stdout or res.stderr)
else:
    res = subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True, text=True)
    print("Repo present —", res.stdout.strip() or "up to date")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)

## § 3 — Mount Drive + Locate the SFT'd LoRA Adapter

**Why we warm-start RL from the SFT'd LoRA, not from the bare base model.**
Starting RL from a random or near-random policy is a waste of compute and
almost always fails: rollouts produce mostly parse failures, all
advantages are essentially noise, and the policy never learns to emit
structured action blocks. SFT puts the model into the rough neighbourhood
of "emit a parseable `<thought>`+`<action>` block that does something
sensible" — RL then *refines* this prior using real env reward.

**Reminder (see the Precondition box at the top):** the adapter this glob
finds must be the **CoT-trained** one — produced by an nb 05 run on the
re-cured `<thought>` dataset. A no-CoT adapter will technically load and
run, but the early reasoning will be poor.

We also use this same Drive folder to checkpoint every $N$ updates so a
Colab disconnect doesn't nuke the run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob
matches = sorted(glob.glob(SFT_ADAPTER_GLOB, recursive=True))
if matches:
    SFT_ADAPTER_PATH = matches[-1]
    print(f"✓ SFT adapter found: {SFT_ADAPTER_PATH}")
    print("  → confirm this adapter was trained on the CoT (re-cured) dataset.")
elif ALLOW_RL_FROM_BASE:
    SFT_ADAPTER_PATH = None
    print("⚠  No SFT adapter found — RL will start from the bare base Gemma.")
    print("    Useful for plumbing tests; not the thesis condition.")
else:
    raise FileNotFoundError(f"No SFT adapter under {SFT_ADAPTER_GLOB}. Run nb 05 first.")

os.makedirs(OUT_DIR, exist_ok=True)
print(f"Run dir: {OUT_DIR}")

## § 4 — Load Gemma + Attach LoRA (initialise from SFT'd adapter)

**What's a LoRA?** *Low-Rank Adapters* are small matrices we slot into the
model's attention and MLP layers. Instead of training all 4 billion
parameters of Gemma, we freeze them and train only the LoRA — typically
0.1–1% as many parameters. This makes RL on a 4B model fit on a free T4,
and it makes the trained delta tiny and easy to ship (the LoRA file is
~50 MB, the base model is ~8 GB).

**What this cell does, step by step:**

1. `FastModel.from_pretrained` — Unsloth's patched 4-bit loader. Returns
   the base Gemma plus a tokenizer that knows the Gemma-4 chat template.
2. `FastModel.get_peft_model` — wraps the base with a fresh, randomly
   initialized LoRA of the same rank/α as the SFT'd one in nb 05.
3. `model.load_adapter(SFT_ADAPTER_PATH, ...)` — overwrites the freshly
   initialised LoRA weights with the trained ones from nb 05. After this
   call, the model behaves *exactly* like the CoT-SFT'd model would have
   at the end of nb 05 — it already emits `<thought>` blocks — and the
   LoRA weights are *trainable*, so RL can adjust them.

**Why we do not merge the SFT LoRA into the base.** We need the *same*
LoRA weights to serve as the trainable policy, while keeping the ability
to *disable* the adapter (§ 5) to get the reference policy for the KL
term. Merging would lose that toggle and force a second model copy into
VRAM.

In [ ]:
from unsloth import FastModel
import torch

torch.cuda.empty_cache()

_t0 = time.time()
model, tokenizer = FastModel.from_pretrained(
    model_name      = MODEL_ID,
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,
    load_in_4bit    = LOAD_IN_4BIT,
    full_finetuning = False,
    token           = HF_TOKEN or None,
)
print(f"Base loaded in {time.time()-_t0:.1f}s")

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    random_state   = SEED,
)
model.print_trainable_parameters()

# ── Warm-start the trainable LoRA from the CoT-SFT adapter ───────────────
if SFT_ADAPTER_PATH is not None:
    try:
        model.load_adapter(SFT_ADAPTER_PATH, adapter_name="default", is_trainable=True)
        print(f"✓ Warm-started LoRA from {SFT_ADAPTER_PATH}")
    except Exception as e:
        print(f"⚠  load_adapter failed ({e}); continuing with the freshly initialized LoRA.")
        print("    RL will still run but starts from a less-informed prior.")

## § 5 — Reference Policy and the KL Term

**Intuition for KL with a temperature analogy.** Imagine you have a
trained chef (the SFT'd model). You want them to get better at one
specific dish (CityLearn control). You give them a critic and let them
adjust their technique. But you don't want them to *forget how to cook
everything else* — or to stop explaining their reasoning. So every time
they change a technique, you ask: "how different is this from your old
way?" Small change → fine. Huge change → push back unless the critic
loves it. That push-back force is the KL term; its strength is $\beta$.

**The technical trick: same model, two modes.** A separate frozen
reference model would need a second copy of Gemma in VRAM. Instead we
exploit a property of LoRA: we can *disable* the adapter at runtime
(`model.disable_adapter()` context manager), and the model temporarily
behaves as if the LoRA had never been added — i.e. as the original base
Gemma. Two forward passes — one with the adapter (the trainable policy),
one without (the reference) — give us both sets of log-probabilities.

**Why a base-Gemma reference is fine for our CoT setup.** `gemma-4-E4B-it`
is an *instruction-tuned* model: given the `make_minimal_prompt` template,
the bare base model already follows the instruction to emit a
`<thought>` block. So the reference is itself **CoT-capable**, and the
KL term anchors the policy's reasoning structure to a model that *does*
reason. This is precisely what keeps the `<thought>` block from
collapsing into filler during RL — the policy is penalised whenever its
reasoning distribution drifts away from a reasoning-capable reference.

*Optional upgrade (not implemented here, kept simple on purpose):* you
could load the CoT-SFT adapter a second time as a frozen named adapter
(`peft` supports multiple adapters) and switch to it for the reference
pass. That anchors to the SFT'd-CoT model specifically rather than to
base Gemma. Do this if KL behaves badly — see `docs/GRPO_EXPLAINED.md`
§ 8.2.

**The cell below is just a sanity check** — it verifies that switching
the adapter on and off actually changes the logits. If `delta ≈ 0`, the
LoRA isn't doing anything and the KL term will be a no-op.

In [ ]:
# Sanity check: disable_adapter() context works and produces different logits.
from src.env import snapshot_state

_probe = "STATE:\nMonth 7, Wed 14:00  |  price=0.540 (PEAK)  |  carbon=0.180 (MID)\nBuildings:\n  B0: SoC= 50.0%  load=2.50 kWh  last_net=+1.00 kWh  solar=HIGH\n  B1: SoC= 40.0%  load=2.00 kWh  last_net=+0.50 kWh  solar=HIGH\n  B2: SoC= 60.0%  load=3.00 kWh  last_net=+1.50 kWh  solar=MID\n"
_msgs  = [{"role":"user", "content": _probe}]
_enc   = tokenizer.apply_chat_template(_msgs, tokenize=True, add_generation_prompt=True,
                                       return_tensors="pt", return_dict=True).to(model.device)

model.eval()
with torch.no_grad():
    logits_pol = model(**_enc).logits[0, -1]
    with model.disable_adapter():
        logits_ref = model(**_enc).logits[0, -1]

delta = (logits_pol - logits_ref).abs().mean().item()
print(f"|logit_policy − logit_ref|_mean = {delta:.4f}  (must be > 0 if LoRA is doing anything)")
assert delta > 1e-6, "LoRA adapter is not active — KL will be a no-op."

## § 6 — Env Factory, Prompt, Action Parser

Three pieces that are shared with every other notebook in the project:

- **`make_colab_env`** — builds a CityLearn 2.6 env with `MERLINReward`
  on the requested building slice. We can call it many times per update,
  one per parallel rollout in the group.
- **`render_state(snapshot)`** — converts a CityLearn observation into the
  human-readable text block the SLM was trained to read. See nb 02/03 for
  the full format.
- **`parse_actions(raw_text, n_buildings)`** — extracts
  `<action building=i>TOKEN</action>` tags from the SLM's output and maps
  the discrete tokens (CHARGE_60, IDLE, …) into floats in $[-1, 1]$.

**Critical:** the RL prompt is `make_minimal_prompt(N_BUILDINGS)` — the
**CoT prompt**, which asks for a `<thought>` block before the action
tags. This is the *same* prompt used at zero-shot eval and at Phase 4
deployment, and (per the Precondition box) the same one nb 05 SFT was
re-run with. One prompt, end to end — no SFT↔eval drift.

In [ ]:
import citylearn
from citylearn.citylearn import CityLearnEnv
from src.env import (
    snapshot_state, MERLINReward, OBSERVATIONS, ACTIVE_ACTIONS,
    TRAINING_BUILDINGS, HELDOUT_BUILDINGS, BUILDINGS,
)
from src.agent import render_state, parse_actions, ACTION_RE, make_minimal_prompt
from src.eval  import evaluate, comparison_table

warnings.filterwarnings("ignore")
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)

SYSTEM_PROMPT = make_minimal_prompt(N_BUILDINGS)   # the CoT prompt (asks for <thought>)
print(f"System prompt: {len(SYSTEM_PROMPT.split())} words.  CoT block requested: "
      f"{'<thought>' in SYSTEM_PROMPT}\n")

def make_colab_env(start: int = 0, end: int = 8758,
                   buildings = None) -> CityLearnEnv:
    env = CityLearnEnv(
        schema="citylearn_challenge_2022_phase_all",
        buildings=buildings if buildings is not None else TRAINING_BUILDINGS,
        central_agent=False,
        active_actions=ACTIVE_ACTIONS,
        active_observations=OBSERVATIONS,
        random_seed=SEED,
        simulation_start_time_step=start,
        simulation_end_time_step=end,
    )
    env.reward_function = MERLINReward(env.get_metadata())
    return env

# Quick smoke: build a 1-step env, render a state, see the format.
_env = make_colab_env(start=4000, end=4001, buildings=TRAIN_SLICE)
_env.reset()
print("--- example rendered state ---")
print(render_state(snapshot_state(_env))[:400])

## § 7 — Rollout Primitive: `rollout_window(t0, slice)`

**This is the most important function in the notebook.** Everything else
is bookkeeping; this is where the policy actually meets the environment.

For each call:

1. Spin up a fresh CityLearn env starting at hour `t0`, restricted to the
   given building slice (3 buildings).
2. For W steps:
   - render the current state as text
   - tokenize the prompt
   - generate a completion **with sampling** (`temperature=0.7, top_p=0.9`)
   - check whether the completion contains a well-formed
     `<thought>…</thought>` block, and how long it is
   - parse the action tokens; if parsing fails, default to all-IDLE and
     apply a fallback penalty
   - step the env with the parsed actions
   - record: prompt tokens, completion tokens, per-step reward, action,
     fallback flag, **thought presence + length**
3. Return all of that plus the summed return.

**Three design choices worth flagging:**

- **Stochastic sampling at rollout time.** If we used greedy decoding the
  G rollouts in a group would be *identical*, group variance would be 0,
  and the advantage would be undefined (Echo Trap on day one). Sampling
  introduces the variance GRPO needs. At eval time we switch back to
  greedy (deterministic) so the reported KPIs are reproducible.
- **Fresh env per rollout.** CityLearn carries state (battery SoC,
  history) between calls; we cannot reuse one env across G rollouts of a
  group. Each rollout gets a brand-new env initialised at the same `t0`.
- **Per-step reward = MERLIN − fallback penalty + CoT bonus.** Parse
  failures get an explicit `FALLBACK_PENALTY` so the gradient discourages
  malformed output. A well-formed `<thought>` block earns the optional
  `COT_BONUS` (default 0.0 — see § 0). With the bonus off, the CoT block
  is shaped only indirectly: it is in the policy-gradient loss like every
  other token, and it is anchored by the KL term to the CoT-capable
  reference.

**CoT instrumentation.** `THOUGHT_RE` extracts the `<thought>` text; we
log `has_thought` (well-formed *and* at least `COT_MIN_WORDS` words) and
`thought_len` per step. These feed the CoT-health plot in § 11 and the
inspection cell in § 11.5 — so you can *watch* whether RL keeps the
reasoning alive.

In [ ]:
# Extracts the reasoning block. DOTALL so it spans newlines; non-greedy so it
# stops at the first </thought>.
THOUGHT_RE = re.compile(r"<thought>(.*?)</thought>", re.IGNORECASE | re.DOTALL)


def inspect_thought(raw: str) -> tuple[bool, int, str]:
    """Return (has_well_formed_thought, word_count, thought_text) for a completion.
    A thought counts as 'well-formed' only if it is closed and >= COT_MIN_WORDS words.
    """
    m = THOUGHT_RE.search(raw)
    if not m:
        return False, 0, ""
    txt = m.group(1).strip()
    wc  = len(txt.split())
    return (wc >= COT_MIN_WORDS), wc, txt


def _build_prompt_ids(state_text: str) -> torch.LongTensor:
    """Build chat-templated prompt input_ids for the current state."""
    # Gemma rejects role='system' — fold SYSTEM_PROMPT into the user turn.
    msgs = [{"role": "user", "content": f"{SYSTEM_PROMPT}\n\nSTATE:\n{state_text}"}]
    enc = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    )
    return enc["input_ids"].to(model.device)


def _generate(prompt_ids: torch.LongTensor, sample: bool) -> torch.LongTensor:
    """Generate a completion. Returns only the NEW token ids (no prompt)."""
    with torch.no_grad():
        out = model.generate(
            input_ids=prompt_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=sample,
            temperature=ROLLOUT_TEMPERATURE if sample else 1.0,
            top_p=ROLLOUT_TOP_P if sample else 1.0,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    return out[0, prompt_ids.shape[1]:].detach()


def rollout_window(t0: int, slice_ids, sample: bool = True) -> dict:
    """One rollout of WINDOW_STEPS steps starting at t0 on the given building slice."""
    env = make_colab_env(start=t0, end=t0 + WINDOW_STEPS, buildings=slice_ids)
    env.reset()

    rec = {"prompts": [], "input_ids": [], "gen_ids": [], "raw": [],
           "rewards": [], "fallbacks": [], "actions": [],
           "has_thought": [], "thought_len": [], "thoughts": []}
    n_b = len(env.buildings)

    FastModel.for_inference(model)
    for step in range(WINDOW_STEPS):
        snap   = snapshot_state(env)
        s_text = render_state(snap)
        p_ids  = _build_prompt_ids(s_text)
        g_ids  = _generate(p_ids, sample=sample)
        raw    = tokenizer.decode(g_ids, skip_special_tokens=True)

        # ── CoT instrumentation ─────────────────────────────────────────
        has_th, th_len, th_txt = inspect_thought(raw)

        # ── action parsing ──────────────────────────────────────────────
        if ACTION_RE.search(raw):
            acts = parse_actions(raw, n_b); fb = False
        else:
            acts = [0.0] * n_b;             fb = True

        _, r_list, term, trunc, _ = env.step([[float(a)] for a in acts])
        # reward = MERLIN  − fallback penalty  + optional CoT bonus
        step_r = (float(sum(r_list))
                  - (FALLBACK_PENALTY if fb else 0.0)
                  + (COT_BONUS if has_th else 0.0))

        rec["prompts"    ].append(s_text)
        rec["input_ids"  ].append(p_ids.squeeze(0).cpu())
        rec["gen_ids"    ].append(g_ids.cpu())
        rec["raw"        ].append(raw)
        rec["rewards"    ].append(step_r)
        rec["fallbacks"  ].append(fb)
        rec["actions"    ].append(acts)
        rec["has_thought"].append(has_th)
        rec["thought_len"].append(th_len)
        rec["thoughts"   ].append(th_txt)
        if term or trunc:
            break

    rec["return"] = float(sum(rec["rewards"]))
    return rec

# Smoke-test one rollout (toy: WINDOW_STEPS=12, so ~60-90 s on T4)
print("Smoke-testing one rollout …")
_t0 = time.time()
_r  = rollout_window(t0=4500, slice_ids=TRAIN_SLICE, sample=True)
print(f"  {len(_r['rewards'])} steps in {time.time()-_t0:.1f}s  |  return={_r['return']:.3f}")
print(f"  fallbacks={sum(_r['fallbacks'])}  |  thought present={sum(_r['has_thought'])}/{len(_r['rewards'])}"
      f"  |  mean thought words={np.mean(_r['thought_len']):.1f}")
if _r['thoughts'] and _r['thoughts'][0]:
    print(f"  example <thought>: {_r['thoughts'][0][:160]!r}")

## § 8 — The GRPO Loss, Concretely

This is the section to read alongside § 0.6 (toy demo) and §§ 3–4 of
[`docs/GRPO_EXPLAINED.md`](../docs/GRPO_EXPLAINED.md). The math is
identical to what the toy demo computed — only now the log-probabilities
come from real torch tensors with gradient hooks attached.

Given a group of G rollouts that all started at the same `t0`, the GRPO
loss for the policy is:

$$
\mathcal{L} \;=\; -\,\mathbb{E}_{g,t}\!\left[ A_g \cdot \log \pi_\theta(c_{g,t}\mid s_{g,t}) \right] \;+\; \beta\,\mathrm{KL}\!\left(\pi_\theta \,\Vert\, \pi_\mathrm{ref}\right)
$$

with $A_g = (R_g - \bar R)/(\sigma_R + \epsilon)$, $R_g = \sum_t r_{g,t}$.

**Concretely, for each `(prompt_ids, gen_ids)` pair from a rollout:**

1. Re-feed `[prompt_ids ; gen_ids]` through the **policy** (LoRA on,
   gradient on) to compute per-completion-token log π_θ(c_t|s_t).
2. Re-feed the same sequence through the **reference** (LoRA off via
   `model.disable_adapter()`, no gradient) to get log π_ref.
3. Sum log-probs over completion tokens → one log π scalar per (g, t).
4. KL term: `(log_pi − log_pi_ref).mean()` over completion tokens.
5. Multiply each step's log-prob by the group advantage `A_g`
   (broadcasting one scalar advantage across all W steps of rollout g),
   sum, negate, add KL.

**`c_{g,t}` is the entire `<thought>…</thought><action>…</action>`
completion — reasoning tokens included.** This is deliberate. Because the
SFT prior is CoT-trained, the `<thought>` tokens carry *real* reasoning
content (not random filler), so putting them in the policy-gradient loss
lets RL refine the *reasoning itself* — not just the final action token.
DeepSeek-R1, RAGEN and ReFT all backprop through the full reasoning
trace for the same reason. The KL term keeps the reasoning anchored to
the CoT-capable reference so it cannot decay into degenerate filler.

**Why two forward passes per (g, t)?** We need log-probs under both the
current policy *and* the frozen reference. Both forward passes use the
*same* model — the only difference is whether the LoRA adapter is active
(`disable_adapter()` context). This is the key trick that lets us avoid
loading a second 4-bit model into VRAM.

**Why we re-feed instead of caching log-probs from the rollout pass.** At
rollout time we ran `model.generate(do_sample=True)` which produced
tokens but didn't return log-probs in a gradient-bearing form. We could
have captured them, but the bookkeeping is fragile and Unsloth's
`for_inference` mode has different attention kernels than the training
mode. Re-feeding is twice as slow but pays for clarity and correctness.

In [ ]:
import torch.nn.functional as F

def _logprobs_of_completion(prompt_ids: torch.Tensor,
                            gen_ids: torch.Tensor,
                            with_grad: bool) -> torch.Tensor:
    """Return per-completion-token log-probabilities under the CURRENT model state.
    Caller controls whether LoRA adapters are enabled via with model.disable_adapter().
    """
    full = torch.cat([prompt_ids, gen_ids], dim=0).unsqueeze(0).to(model.device)
    ctx  = torch.enable_grad() if with_grad else torch.no_grad()
    with ctx:
        logits = model(full).logits[0]   # [T, V]
    # Shift so logits at position i predict token at i+1
    p_len = prompt_ids.shape[0]
    g_len = gen_ids.shape[0]
    # Logits aligned with completion tokens are at positions [p_len-1 .. p_len+g_len-2]
    comp_logits = logits[p_len-1 : p_len-1 + g_len]                  # [g_len, V]
    log_probs   = F.log_softmax(comp_logits.float(), dim=-1)
    targets     = gen_ids.to(model.device).unsqueeze(-1)              # [g_len, 1]
    return log_probs.gather(-1, targets).squeeze(-1)                  # [g_len]


def grpo_loss(group_records: list[dict]):
    """Compute the GRPO loss for one group of G rollouts sharing a t0."""
    returns = torch.tensor([r["return"] for r in group_records], dtype=torch.float32)
    adv     = (returns - returns.mean()) / (returns.std() + 1e-6)     # [G]

    pg_terms, kl_terms = [], []
    FastModel.for_training(model)

    for g, rec in enumerate(group_records):
        for p_ids, g_ids in zip(rec["input_ids"], rec["gen_ids"]):
            if g_ids.numel() == 0:
                continue
            # log π over the WHOLE completion (thought + action tokens)
            lp_pol = _logprobs_of_completion(p_ids, g_ids, with_grad=True)
            with model.disable_adapter():
                lp_ref = _logprobs_of_completion(p_ids, g_ids, with_grad=False)

            # Policy gradient term: − A_g · Σ_t log π(c_t|s_t)
            pg_terms.append(-adv[g] * lp_pol.sum())
            # KL term: mean over completion tokens
            kl_terms.append((lp_pol - lp_ref).mean())

    if not pg_terms:
        return None, {"pg": 0.0, "kl": 0.0, "loss": 0.0}

    pg = torch.stack(pg_terms).mean()
    kl = torch.stack(kl_terms).mean()
    loss = pg + KL_BETA * kl
    return loss, {"pg": pg.item(), "kl": kl.item(), "loss": loss.item()}

## § 9 — Training Loop

The outer loop wires everything together. Per update:

1. Sample a starting hour `t0` from the seasonal pool (uniform over the
   four seasons → policy doesn't overfit to summer).
2. Collect G rollouts of W steps from that `t0` (§ 7).
3. Compute the GRPO loss (§ 8), backprop, clip gradients, step optimizer.
4. Append a row to a JSONL log on Drive — including the **CoT-health**
   metrics (thought-presence rate, mean thought length).
5. Every `EVAL_EVERY` updates, run a deterministic 1-week greedy eval on
   the training slice and log the KPI.
6. Every `CHECKPOINT_EVERY` updates, save the LoRA adapter to Drive.

**What healthy training looks like, qualitatively:**

- *Update 1–10*: mean return very noisy, KL near 0, fallback rate may be
  non-trivial (5–20%) if the SFT prior was weak in the season we sampled;
  thought-presence should already be high (~90%+) because the SFT prior
  is CoT-trained.
- *Update 10–50*: mean return trending up (less negative); KL growing
  steadily; fallback rate falling toward 0–2%; thought-presence stays
  high.
- *Update 50+*: mean return plateauing; KL plateauing too (good); group
  std stays > 0 (no Echo Trap); thought-presence still ~90%+.

**Four failure modes to watch for** (all visible in the § 11 plots):

- **Echo Trap**: group std collapses to 0 → no learning signal. Fix:
  raise rollout temperature, raise KL β, shorten W.
- **KL blow-up**: KL grows monotonically and unboundedly → policy
  degrades. Fix: double KL β; reload from last good checkpoint.
- **Fallback collapse**: parse failure rate climbs above ~5% and stays
  there. Fix: raise `FALLBACK_PENALTY`; raise KL β.
- **CoT collapse**: thought-presence rate falls, or mean thought length
  shrinks toward `COT_MIN_WORDS`. The reasoning is degenerating into
  filler. Fix: raise KL β (stronger anchor to the CoT reference); as a
  last resort enable a small `COT_BONUS` (§ 0).

See `docs/GRPO_EXPLAINED.md` § 8 for a fuller diagnosis-and-fix table.

In [ ]:
from torch.optim import AdamW

trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable, lr=LEARNING_RATE)

rng = random.Random(SEED)
_season_keys = list(T0_POOL_SEASONS.keys())

def sample_t0() -> int:
    season = rng.choice(_season_keys)
    lo, hi = T0_POOL_SEASONS[season]
    hi = min(hi, 8759 - WINDOW_STEPS - 1)
    return rng.randint(lo, hi)


def quick_eval(label: str) -> dict:
    """Deterministic 1-week eval on TRAIN_SLICE — cheap probe between updates."""
    env = make_colab_env(start=EVAL_START, end=EVAL_START + EVAL_LEN - 1,
                         buildings=TRAIN_SLICE)
    env.reset()
    n_b, n_fb, total_r, n_th = len(env.buildings), 0, 0.0, 0
    done, t = False, 0
    while not done:
        snap   = snapshot_state(env)
        s_text = render_state(snap)
        p_ids  = _build_prompt_ids(s_text)
        g_ids  = _generate(p_ids, sample=False)        # greedy
        raw    = tokenizer.decode(g_ids, skip_special_tokens=True)
        has_th, _, _ = inspect_thought(raw)
        n_th  += int(has_th)
        if ACTION_RE.search(raw):
            acts = parse_actions(raw, n_b)
        else:
            acts = [0.0]*n_b; n_fb += 1
        _, r_list, term, trunc, _ = env.step([[float(a)] for a in acts])
        total_r += float(sum(r_list)); t += 1
        done = bool(term or trunc)
    res = evaluate(env, label=label)
    return {"label": label, "phase1": res.phase1, "combined": res.combined,
            "reward_sum": total_r, "fallbacks": n_fb, "cot_rate": n_th / max(1, t)}


log_path = f"{OUT_DIR}/train_log.jsonl"
Path(log_path).parent.mkdir(parents=True, exist_ok=True)
print(f"Logging to {log_path}")
print(f"Starting GRPO training — {UPDATES} updates, G={GROUP_SIZE}, W={WINDOW_STEPS}\n")

history = []
for update in range(1, UPDATES + 1):
    upd_t0 = time.time()
    t0     = sample_t0()

    # ── 1. Collect G rollouts from the same t0 ───────────────────────────
    group = []
    for g in range(GROUP_SIZE):
        group.append(rollout_window(t0=t0, slice_ids=TRAIN_SLICE, sample=True))

    returns = [r["return"] for r in group]
    n_steps_tot = sum(len(r["rewards"]) for r in group)
    fb_rate  = sum(sum(r["fallbacks"])   for r in group) / max(1, n_steps_tot)
    cot_rate = sum(sum(r["has_thought"]) for r in group) / max(1, n_steps_tot)
    tl_all   = [tl for r in group for tl in r["thought_len"]]
    thought_len_mean = float(np.mean(tl_all)) if tl_all else 0.0

    # ── 2. GRPO loss + step ──────────────────────────────────────────────
    optimizer.zero_grad(set_to_none=True)
    loss, info = grpo_loss(group)
    if loss is not None:
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP)
        optimizer.step()

    # ── 3. Log ───────────────────────────────────────────────────────────
    rec = {
        "update":       update,
        "t0":           t0,
        "returns":      returns,
        "return_mean":  float(np.mean(returns)),
        "return_std":   float(np.std(returns)),
        "fallback_rate": fb_rate,
        "cot_rate":      cot_rate,
        "thought_len_mean": thought_len_mean,
        "wall_s":       time.time() - upd_t0,
        **info,
    }
    history.append(rec)
    with open(log_path, "a") as f:
        f.write(json.dumps(rec) + "\n")

    print(f"upd {update:3d}/{UPDATES}  t0={t0:5d}  R̄={rec['return_mean']:+.3f}±{rec['return_std']:.3f}  "
          f"loss={info['loss']:+.4f}  KL={info['kl']:+.3f}  fb={fb_rate*100:.0f}%  "
          f"CoT={cot_rate*100:.0f}%({thought_len_mean:.0f}w)  ({rec['wall_s']:.0f}s)")

    # ── 4. Eval probe ────────────────────────────────────────────────────
    if update % EVAL_EVERY == 0 or update == UPDATES:
        ev = quick_eval(label=f"rl_upd{update}")
        print(f"  ↳ eval week (greedy): Phase I={ev['phase1']:.4f}  Combined={ev['combined']:.4f}  "
              f"fb={ev['fallbacks']}/{EVAL_LEN}  CoT={ev['cot_rate']*100:.0f}%")
        with open(log_path, "a") as f:
            f.write(json.dumps({"eval": ev, "update": update}) + "\n")

    # ── 5. Checkpoint to Drive ───────────────────────────────────────────
    if update % CHECKPOINT_EVERY == 0 or update == UPDATES:
        ck = f"{OUT_DIR}/lora_adapter_upd{update}"
        model.save_pretrained(ck)
        tokenizer.save_pretrained(ck)
        print(f"  ↳ checkpoint → {ck}")

print("\nTraining done.")

## § 10 — Final Evaluation: RL vs RBC vs No-Control

Run the **full-year** (or in toy mode, 1-week) KPI evaluation on:

- **Training slice** `[0, 1, 2]` — should beat RBC and No-Control. This
  answers RQ1 (can the SLM compete with rule-based and RL baselines?).
- **Held-out slice** `[3, 4, 5]` — same buildings the model has never
  trained on. Tells us how much the policy generalises. Gap to training
  KPI ≤ 20% is the thesis success criterion (RQ2).

The `Phase I = (C + G) / 2` row in the challenge table is the canonical
thesis metric. *Lower is better* — values < 1.0 mean the agent reduces
cost+carbon relative to the no-battery baseline.

Toy mode (default) runs 1 week → 168 env steps × ~5s/call ≈ 14 minutes
per policy. For the real thesis number, set `FULL_SCALE = True` in § 0
and rerun this cell — full year is 8,760 steps ≈ 12 hours per policy on
T4.

In [ ]:
from citylearn.agents.rbc import BasicBatteryRBC

EVAL_LEN_FINAL = 168 if not FULL_SCALE else 8758

def run_policy(label, slice_ids, length=EVAL_LEN_FINAL):
    env = make_colab_env(start=0, end=length - 1, buildings=slice_ids)
    env.reset()
    n_b, n_fb, n_th = len(env.buildings), 0, 0
    done, t = False, 0
    t0 = time.time()
    while not done:
        s_text = render_state(snapshot_state(env))
        p_ids  = _build_prompt_ids(s_text)
        g_ids  = _generate(p_ids, sample=False)
        raw    = tokenizer.decode(g_ids, skip_special_tokens=True)
        has_th, _, _ = inspect_thought(raw); n_th += int(has_th)
        if ACTION_RE.search(raw):
            acts = parse_actions(raw, n_b)
        else:
            acts = [0.0]*n_b; n_fb += 1
        _, _, term, trunc, _ = env.step([[float(a)] for a in acts])
        done = bool(term or trunc); t += 1
    print(f"[{label}] {t} steps  {time.time()-t0:.0f}s  fallbacks={n_fb}/{t}  CoT={n_th/max(1,t)*100:.0f}%")
    return env

def run_rbc(slice_ids, length=EVAL_LEN_FINAL):
    env = make_colab_env(start=0, end=length-1, buildings=slice_ids)
    BasicBatteryRBC(env=env).learn(episodes=1)
    return env

def run_noop(slice_ids, length=EVAL_LEN_FINAL):
    env = make_colab_env(start=0, end=length-1, buildings=slice_ids)
    env.reset(); n_b = len(env.buildings)
    done = False
    while not done:
        _, _, term, trunc, _ = env.step([[0.0] for _ in range(n_b)])
        done = bool(term or trunc)
    return env

print("── Evaluating on TRAIN_SLICE [0,1,2] ──")
env_noop  = run_noop(TRAIN_SLICE)
env_rbc   = run_rbc (TRAIN_SLICE)
env_rl    = run_policy("rl-train", TRAIN_SLICE)

print("\n── Evaluating on HELDOUT_SLICE [3,4,5] (generalization) ──")
env_rl_h  = run_policy("rl-heldout", HELDOUT_SLICE)
env_rbc_h = run_rbc(HELDOUT_SLICE)

results = [
    evaluate(env_noop, "No-Control [train]"),
    evaluate(env_rbc,  "RBC [train]"),
    evaluate(env_rl,   "GRPO-SLM [train]"),
    evaluate(env_rbc_h,"RBC [heldout]"),
    evaluate(env_rl_h, "GRPO-SLM [heldout]"),
]
challenge_df, zne_df = comparison_table(results)
print("\nChallenge scores — lower is better; No-Control is the reference.")
display(challenge_df.round(4))
print("\nZNE + self-consumption:")
display(zne_df[["solar generation (kWh)", "grid import (kWh)",
                "ZNE ratio (solar / import)", "self-consumption ratio"]].round(4))

## § 11 — Training Curves: How to Read Them

Four plots. Each one tells you about a different failure mode.

### Plot 1 — Policy return per update
**X-axis**: optimizer update number. **Y-axis**: mean window-return
across the G rollouts of that update; shaded band is ±1 std across the
group.

- *Healthy*: line trends **upward** (less negative); shaded band stays
  noticeably wide throughout training.
- *Echo Trap*: line plateaus *and* shaded band collapses to a thin
  ribbon. Group variance is gone, advantages are uninformative.
- *Catastrophic forgetting*: line drops sharply downward, usually
  alongside the KL plot spiking.

### Plot 2 — KL(π ‖ π_ref)
**Y-axis**: per-token KL divergence between the policy and the
(LoRA-disabled) reference.

- *Healthy*: starts ~0, grows monotonically for the first ~30% of
  training, then **plateaus** in the 5–20 nats/token range.
- *KL blow-up*: grows linearly forever. Double KL β; if persists, reload
  last checkpoint.
- *KL never grows*: policy isn't actually moving. Check LR ≠ 0 and the
  LoRA adapter is active (§ 5 sanity check).

### Plot 3 — Action parse-failure rate
**Y-axis**: % of steps where the SLM's output could not be parsed as an
action block (fell back to all-IDLE).

- *Healthy*: low single digits (0–5%) at the start; falls toward 0%.
- *Fallback collapse*: rate climbs and stays above 5%. Raise
  `FALLBACK_PENALTY`, raise KL β.

### Plot 4 — CoT health (new)
**Left Y-axis** (blue): % of steps whose completion contains a
well-formed `<thought>` block (closed, ≥ `COT_MIN_WORDS` words).
**Right Y-axis** (green): mean `<thought>` length in words.

- *Healthy*: presence stays high (~90%+) and length holds roughly steady
  — RL is refining the *content* of the reasoning, not eroding it.
- *CoT collapse*: presence drifts down, or length shrinks toward
  `COT_MIN_WORDS`. The reasoning is degenerating into filler. Raise KL β
  (the dominant fix — anchors harder to the CoT-capable reference); as a
  last resort enable a small `COT_BONUS` in § 0.

### What about the eval table (below the plots)?
That table tracks Phase I / Combined KPI on a deterministic greedy
1-week eval at every `EVAL_EVERY` updates, plus the greedy CoT rate.
Window return is what the policy *optimises*; KPI is what the thesis is
*judged* on. They should move in the same direction.

In [ ]:
import matplotlib.pyplot as plt

log_rows = [json.loads(l) for l in open(log_path)]
train_rows = [r for r in log_rows if "update" in r and "return_mean" in r]
eval_rows  = [r["eval"] for r in log_rows if "eval" in r]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
x = [r["update"] for r in train_rows]

# Plot 1 — return
axes[0].plot(x, [r["return_mean"] for r in train_rows], label="mean return (window)")
axes[0].fill_between(x,
    [r["return_mean"] - r["return_std"] for r in train_rows],
    [r["return_mean"] + r["return_std"] for r in train_rows], alpha=0.2)
axes[0].set_xlabel("update"); axes[0].set_ylabel("window return")
axes[0].set_title("Policy return per update"); axes[0].grid(alpha=0.3); axes[0].legend()

# Plot 2 — KL
axes[1].plot(x, [r["kl"] for r in train_rows], color="tab:orange")
axes[1].set_xlabel("update"); axes[1].set_ylabel("KL [nats / completion token]")
axes[1].set_title("KL to reference"); axes[1].grid(alpha=0.3)

# Plot 3 — fallback rate
axes[2].plot(x, [100*r["fallback_rate"] for r in train_rows], color="tab:red")
axes[2].set_xlabel("update"); axes[2].set_ylabel("% steps with parse failure")
axes[2].set_title("Action-parse failure rate"); axes[2].grid(alpha=0.3)

# Plot 4 — CoT health (twin axis: presence % and mean thought length)
axes[3].plot(x, [100*r.get("cot_rate", 0) for r in train_rows],
             color="tab:blue", label="thought present %")
axes[3].set_xlabel("update"); axes[3].set_ylabel("% steps with <thought>", color="tab:blue")
axes[3].set_ylim(0, 105); axes[3].tick_params(axis="y", labelcolor="tab:blue")
axes[3].set_title("CoT health"); axes[3].grid(alpha=0.3)
ax3b = axes[3].twinx()
ax3b.plot(x, [r.get("thought_len_mean", 0) for r in train_rows],
          color="tab:green", linestyle="--", label="mean thought words")
ax3b.set_ylabel("mean <thought> words", color="tab:green")
ax3b.tick_params(axis="y", labelcolor="tab:green")

plt.tight_layout(); plt.show()

if eval_rows:
    print("\nEval-week KPI progression (greedy, TRAIN_SLICE):")
    import pandas as pd
    display(pd.DataFrame(eval_rows).set_index("label").round(4))

## § 11.5 — Inspect the Chain-of-Thought

The plots in § 11 tell you *whether* the `<thought>` block survives RL.
This cell tells you *whether it makes sense*. It runs a short greedy
rollout and prints, for each step, the rendered state, the SLM's
`<thought>` rationale, and the parsed actions side by side.

**What to look for** — a healthy rationale should reference the signals
that actually drive the decision: current price (LOW/PEAK), solar
(NONE/LOW/HIGH), battery SoC, and the load. "B0: PEAK price and SoC high
→ discharge" is good. "ok ok ok" or an empty thought is CoT collapse.
This is the manual evidence for go/no-go checklist item 7 (§ 13) and for
thesis RQ3 (interpretable rationales).

In [ ]:
INSPECT_STEPS = 6
INSPECT_T0    = EVAL_START

env_i = make_colab_env(start=INSPECT_T0, end=INSPECT_T0 + INSPECT_STEPS, buildings=TRAIN_SLICE)
env_i.reset()
n_b = len(env_i.buildings)

print("="*78)
print(f"COT INSPECTION — {INSPECT_STEPS} greedy steps from t0={INSPECT_T0}")
print("="*78)
for step in range(INSPECT_STEPS):
    snap   = snapshot_state(env_i)
    s_text = render_state(snap)
    p_ids  = _build_prompt_ids(s_text)
    g_ids  = _generate(p_ids, sample=False)            # greedy = deterministic
    raw    = tokenizer.decode(g_ids, skip_special_tokens=True)
    has_th, th_len, th_txt = inspect_thought(raw)
    acts   = parse_actions(raw, n_b) if ACTION_RE.search(raw) else [0.0]*n_b

    print(f"\n── step {step}  ({s_text.splitlines()[0]}) " + "─"*20)
    for ln in s_text.splitlines()[1:]:
        print(f"   {ln}")
    flag = f"✓ {th_len}w" if has_th else "✗ MISSING/degenerate"
    print(f"   <thought> [{flag}]: {th_txt if th_txt else '(none)'}")
    print(f"   → actions: {[round(a,2) for a in acts]}")

    env_i.step([[float(a)] for a in acts])

print("\n" + "="*78)
print("Manual check: do the rationales cite price / solar / SoC / load sensibly?")
print("If yes → RQ3 evidence. If thoughts are empty/degenerate → see § 11 CoT plot.")

## § 12 — Persist Final Adapter to Drive (canonical name)

The final adapter goes to `MyDrive/eclipse-thesis/rl_adapters/<RUN_NAME>/`.
This is the artefact you load into both Phase 4 agents (α on B0-2, β on
B3-5) — same LoRA, no retraining.

In [ ]:
FINAL_DIR = f"{DRIVE_ROOT}/rl_adapters/{RUN_NAME}"
os.makedirs(FINAL_DIR, exist_ok=True)
model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"✓ Final adapter saved: {FINAL_DIR}")
print("  This is the artifact to load for Phase 4 dual-agent deployment.")

## § 13 — Go / No-Go Checklist before Phase 4

**Precondition recap:** this checklist assumes nb 07 was run on a
CoT-capable SFT adapter (nb 05 re-run on the re-cured `<thought>`
dataset). If you ran RL from a no-CoT adapter, the rationale criteria
(#7) cannot pass — fix the dataset and re-run.

Before promoting this LoRA into the two Phase 4 agents (α on B0–2, β on B3–5), confirm all of the following on the **full-year** rerun of § 10 (set `FULL_SCALE = True`, `EVAL_LEN_FINAL = 8758`, rerun §§ 10–11):

| # | Criterion | Threshold | Where to check |
|---|---|---|---|
| 1 | Phase I KPI vs SAC on training buildings | ≤ 1.3 × SAC (≥ 70 % of SAC perf — CLAUDE.md gate) | § 10 challenge table |
| 2 | GRPO-SLM beats RBC and No-Control on Phase I | strict on both | § 10 |
| 3 | Generalization gap (train vs heldout) | ≤ 20 % relative degradation | § 10 |
| 4 | Action-parse fallback rate, full-year greedy | ≤ 2 % | § 10 print |
| 5 | KL(π ‖ π_ref) at end of training | ≤ 20 nats/seq, plateaued (no monotonic blow-up) | § 11 plot 2 |
| 6 | Reward variance non-zero through training (no "Echo Trap" — RAGEN) | std stays > 0.1 × \|mean\| through last 25 % of training | § 11 plot 1 |
| 7 | CoT survives RL — thought present + rationales sensible | presence ≥ 90 % full-year greedy; 20 sampled `<thought>` blocks cite price/solar/SoC/load plausibly | § 11 plot 4 + § 11.5 |
| 8 | Checkpoint reproducibility | reloading from `FINAL_DIR` reproduces the § 10 KPI within 1 % | smoke rerun |

Fail any of #1–#3 → iterate before Phase 4. Tuning levers in priority order:
1. **Window length `W`** — too short → policy can't see consequences; too long → return variance explodes.
2. **`KL_BETA`** — halve if KL is exploding; double if reward collapses to a single mode *or* if the § 11 CoT plot shows reasoning decaying.
3. **`LEARNING_RATE`** — drop 2× if loss is noisy.
4. **`FALLBACK_PENALTY`** — raise if parse failures persist past update ~50.
5. **`COT_BONUS`** — last resort if CoT keeps decaying despite a higher KL β: set a small positive value (0.03–0.08) so a well-formed `<thought>` earns a per-step reward bonus.

## § 14 — Reading Order for the Papers Cited

If you want to understand the field at the depth a thesis defence will
demand, read these in order. The full annotated version with
section-level pointers is in
[`docs/GRPO_EXPLAINED.md`](../docs/GRPO_EXPLAINED.md) § 9; the short
version is:

### Tier 1 (read in full)
1. **DeepSeekMath / GRPO** — Shao et al. 2024.
   [arXiv:2402.03300](https://arxiv.org/abs/2402.03300). The algorithm.
   § 4 is the GRPO definition. § 5.2 has the ablation justifying G=4–8.
2. **RAGEN / StarPO** — Wang et al. 2025.
   [arXiv:2504.20073](https://arxiv.org/abs/2504.20073). The closest
   paper to our setup. Read §§ 1–4. "Echo Trap" diagnosis + StarPO-S
   stabilizers are what this notebook's failure-mode discussion is based
   on.

### Tier 2 (read sections)
3. **ReFT** — Trung et al. 2024 (ACL).
   [arXiv:2401.08967](https://arxiv.org/abs/2401.08967). § 3 is the
   SFT-warmup → PPO/RL recipe on CoT rationales. Conceptual blueprint
   for our nb 05 → nb 07 pipeline.
4. **ArCHer** — Zhou et al. 2024 (ICML).
   [arXiv:2402.19446](https://arxiv.org/abs/2402.19446). § 2 (problem
   formulation) and § 3 (hierarchical TD critic). Useful for
   understanding *what we're not doing* (step-level credit assignment).

### Tier 3 (skim)
5. **RLOO** — Ahmadian et al. 2024 (ACL).
   [arXiv:2402.14740](https://arxiv.org/abs/2402.14740). Justifies the
   critic-free family. Algorithmically close to GRPO.
6. **MERLIN reward** — Nweye et al. 2024 (Applied Energy 358).
   [doi:10.1016/j.apenergy.2023.121958](https://doi.org/10.1016/j.apenergy.2023.121958).
   Already cited in your thesis. Table 3 has our reward parameterisation.

### Tier 4 (further reading)
7. **GiGPO / SALT** — recent (2025) step-level credit assignment for
   multi-turn agentic RL. Out of scope for this notebook but useful if
   you ever upgrade beyond window-level credit.

### Mapping to thesis chapters

| Notebook concept | Thesis chapter / section |
|---|---|
| The MDP framing in § 0.5 / § 2 of the doc | Background — RL in CityLearn |
| Policy gradients, baselines | Background — Policy Gradient Methods |
| GRPO formal definition | Methods — RL Fine-Tuning |
| Windowed adaptation | Methods — Multi-Step Credit Assignment |
| CoT-in-the-loss design | Methods — Reasoning Trace & Interpretability (RQ3) |
| Worked example (§ 0.6) | Methods — Worked Example (appendix) |
| Algorithm comparison (`docs/GRPO_EXPLAINED.md` § 6) | Methods — Algorithm Selection |
| Hyperparameter table (`docs/GRPO_EXPLAINED.md` § 7) | Methods — Hyperparameters |
| Pitfalls (`docs/GRPO_EXPLAINED.md` § 8) | Results — Training Dynamics |